In [72]:
import numpy as np
import cv2
import json
import csv
import pathlib
from pathlib import Path
import os
import re
import pickle
import scipy.spatial.transform
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image as IPImage


complete_corner_mapping = {
    "AB": ["A","B","D","C"],
    "BC": ["B","C","A","D"],
    "CD": ["C","D","B","A"],
    "DA": ["D","A","C","B"],
    "AD": ["A","D","B","C"],
    "DC": ["D","C","A","B"],
    "CB": ["C","B","D","A"],
    "BA": ["B","A","C","D"]
}

clockwise_rotation = ["AB","BC","CD","DA"]

default_corner_order = "BC"

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.0001)

# Calibration of N Cameras

**Note**: This calibration procedure requires
- A rectangular checkerboard with one recognizable side:
    ```
    A             B
      |-|-|-|-|-|
      | | | | | |
      | | | | | |
    D |-|-|-|-|-| C
      visual clue
    ```
- A manually created file `video_2_frame_number.json` which annotates for each video the frame-sequences during which the checkerboard is visible in this video:
    ```json
    {
        "cam-2": {
        "sequences": [
            [129, 344],
            [1777, 2876]
        ],
        "best_sequences_idx": [1]
        },
        "cam-3": {
            "sequences": [
                [330, 657], 
                [1355, 1552], 
                [1775, 2372], 
                [4031, 4272]
            ],
            "best_sequences_idx": [2,3] 
        },
        ...
    }
    ```
- A manually created file `filename_2_upper_side.json` which annotates during which frame-sequences which side of the checkerboard points upwards:
    ```json
    {
        "video_0.mp4": [
            {
                "start": 0,
                "end": 132,
                "upside": "BC"
            },
            ...
        ],
        ...
    }
    ```
    The specified sequences also determine which frames are gonna be used for calibration.


## Utility

### Drawing

In [73]:
def draw_corners_with_order(image, corners):
    """Custom drawing function for detected chessboard corners."""
    corners = corners.reshape(-1, 2)  # Flatten the corners for easier indexing

    # Copy the image to draw on
    image_copy = image.copy()

    # Define font, color, and scale for drawing text
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.5
    color = (0, 0, 255)  # Red color for the text
    thickness = 1

    # Draw the corner points with their order number
    for i, corner in enumerate(corners):
        corner_position = tuple(corner.astype(int))
        cv2.circle(
            image_copy, corner_position, 5, (0, 255, 0), -1
        )  # Draw a small circle at the corner
        cv2.putText(
            image_copy,
            str(i + 1),
            corner_position,
            font,
            font_scale,
            color,
            thickness,
            cv2.LINE_AA,
        )

    return image_copy

### Detection

In [74]:
def dict_from_dict_or_json_file(file_or_dict):
    if type(file_or_dict) is dict:
        return file_or_dict
    else: 
        with open(file_or_dict) as jf:
            file_content = json.load(jf)
        return file_content

In [75]:
def detect_checkerboards(outdir, video_name, cbrows, cbcols, enhance=True, slope_change_threshold_deg=15, start_row: int = 0):
    """Detect checkerboards in images from files.csv, track results in index.json.
    Uses files.csv for frame numbers and updates it with upper_side labels.
    Can resume from a given row index in files.csv by passing start_row.
    """
    outdir = Path(outdir)

    # Load index.json
    index_path = outdir / 'index.json'
    with index_path.open() as f:
        index = json.load(f)

    # Load files.csv for this video
    files_csv = Path(index['index_files'][video_name])
    files_data = []
    with files_csv.open() as f:
        reader = csv.DictReader(f)
        files_data = list(reader)

    # criteria for corner detection
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

    # prepare object points template
    objp = np.zeros((cbcols*cbrows,3), np.float32)
    objp[:,:2] = np.mgrid[0:cbrows,0:cbcols].T.reshape(-1,2)

    # Arrays to store object points and image points from all the images.
    objpoints = {} # 3d point in real world space
    imgpoints = {} # 2d points in image plane.

    prev_angle = None
    changed_frames = []  # list of (filename, frame_number, row_idx)

    def compute_angle_deg(p1, p2):
        return np.degrees(np.arctan2(p2[1] - p1[1], p2[0] - p1[0]))

    video_dir = outdir / video_name
    vis_dir = video_dir / 'visualizations'
    vis_dir.mkdir(parents=True, exist_ok=True)

    # Detection phase
    print(f"\nProcessing {len(files_data)} frames from row {start_row}...")
    for i in range(start_row, len(files_data)):
        row = files_data[i]
        imgfile = Path(row['file_loc'])
        frame_num = int(row['frame'])
        
        # Update status frequently
        if i % 10 == 0:
            index['status'] = {"step": "detect", "video": video_name, "frame": frame_num, "row": i}
            with index_path.open('w') as f:
                json.dump(index, f, indent=2)
            print(f"Processing frame {frame_num} ({i+1}/{len(files_data)})...", end='\r')

        img = cv2.imread(str(imgfile))
        if img is None:
            print(f"\nWarning: Could not read image {imgfile}")
            continue

        if enhance:
            # Apply Gaussian blur to reduce noise
            img = cv2.GaussianBlur(img, (5,5), 0)
            
            # Convert to grayscale
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # Adaptive histogram equalization
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            gray = clahe.apply(gray)
        else:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Find the chess board corners
        ret, corners = cv2.findChessboardCorners(gray, (cbrows,cbcols), None)

        # If found, add object points, image points (after refining them)
        if ret == True:
            objpoints[imgfile.name] = [objp.tolist()]
            corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
            imgpoints[imgfile.name] = [corners2.tolist()]

            # Draw and save the corners
            img_vis = cv2.drawChessboardCorners(img.copy(), (cbrows,cbcols), corners2, ret)
            vis_path = vis_dir / f"{imgfile.name}.png"
            cv2.imwrite(str(vis_path), img_vis)

            # Get points for computing board orientation (outer edges middle points)
            points = corners2.reshape(cbrows, cbcols, 2)
            left_mid = (points[cbrows//2, 0] + points[cbrows//2-1, 0]) / 2
            right_mid = (points[cbrows//2, -1] + points[cbrows//2-1, -1]) / 2
            
            # Compute angle of the edge that's meant to face upwards
            angle = compute_angle_deg(left_mid, right_mid)
            
            # If this is a significant angle change from previous frame, record it
            if prev_angle is not None and abs(angle - prev_angle) > slope_change_threshold_deg:
                changed_frames.append((imgfile.name, frame_num, i))
            prev_angle = angle

    print(f"\nDetection complete. Found corners in {len(imgpoints)} frames.")

    # Save detected points
    objpoints_path = video_dir / 'objpoints.json'
    imgpoints_path = video_dir / 'imgpoints.json'
    
    with objpoints_path.open('w') as f:
        json.dump(objpoints, f)
    with imgpoints_path.open('w') as f:
        json.dump(imgpoints, f)

    # Update index.json to track imgpoints
    if 'imgpoints' not in index:
        index['imgpoints'] = {}
    index['imgpoints'][video_name] = str(imgpoints_path)
    index['status'] = {"step": "detect_complete", "video": video_name}
    with index_path.open('w') as jf:
        json.dump(index, jf, indent=2)

    # Dictionary for corner order labels
    complete_corner_mapping = {"AB", "BC", "CD", "DA", "BA", "CB", "DC", "AD"}

    # Prompt user for upside labels where angle changed
    if len(changed_frames) > 0:
        print("\nDetected changes in upwards-facing edge in these frames:")
        clear_output(wait=True)  # Clear previous outputs
        
        current_upper = None
        for filename, frame_num, row_idx in changed_frames:
            vis_path = str(video_dir / 'visualizations' / f"{filename}.png")
            
            # Display image inline in notebook
            print(f"\nFrame {frame_num} (showing visualization of detected corners):")
            display(Image(filename=vis_path))
            
            user_input = input(f"Enter upwards-facing side for frame {frame_num} (e.g. 'BC' or 'AB'): ").strip().upper()
            if len(user_input) == 0:
                print("Empty input - using default 'BC'")
                user_input = "BC"
            if len(user_input) > 2:
                print(f"Warning: truncating '{user_input}' to first two letters")
                user_input = user_input[:2]
            if user_input not in complete_corner_mapping:
                print(f"Warning: '{user_input}' not in known corner mappings. Saving anyway.")

            # Fill files_data from this row until the next change with this upper side
            # We update the row at row_idx and subsequent rows until next changed_frame row
            next_idx = None
            for _, _, r in changed_frames:
                if r > row_idx:
                    next_idx = r
                    break
            end_idx = next_idx if next_idx is not None else len(files_data)
            for r in range(row_idx, end_idx):
                files_data[r]['upper_side'] = user_input

            clear_output(wait=True)  # Clear previous frame/input
                
    # Write updated files.csv with upper_side labels
    with files_csv.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(files_data[0].keys()))
        writer.writeheader()
        writer.writerows(files_data)

    return objpoints, imgpoints

In [76]:
def enhance_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    # enhanced = cv2.equalizeHist(gray)
    # enhanced = cv2.GaussianBlur(enhanced, (5, 5), 0)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # Sharpening
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    enhanced = cv2.filter2D(gray, -1, kernel)
    return enhanced

In [77]:
def extract_from_video(
    videos: list[Path],
    video_2_frame_number,
    out_dir: Path,
    start_video: str = None,
    start_frame: int = 0,
):
    """Extract frames for multiple videos and write index.json and files.csv.

    Can resume extraction by passing start_video and start_frame.
    The function updates index.json['status'] while running so a caller can resume later.
    Adds a 'category' column with value 'original' for extracted frames.
    """
    video_2_frame_number = dict_from_dict_or_json_file(video_2_frame_number)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    json_out_path = out_dir / "index.json"
    # If an index exists already, load it and preserve previous data
    if json_out_path.exists():
        with json_out_path.open() as f:
            json_index = json.load(f)
    else:
        json_index = {"frame_folders": [], "index_files": {}, "image_sizes": {}, "image_count": 0}

    # Ensure status field exists
    json_index.setdefault('status', {})

    resume_mode = start_video is not None

    for video_path in videos:
        if not video_path.exists():
            raise Exception(f"Input video does not exist: {video_path}")

        print(f"processing video: {video_path}")
        video_name = video_path.stem

        # read one frame to get vheight and vwidth
        capture = cv2.VideoCapture(str(video_path))
        try:
            if not capture.isOpened():
                raise ValueError(f"couldn't open video: {video_path}")
            success, frame = capture.read()
            if not success:
                raise ValueError(f"video {video_path} couldn't be read.")
            vheight, vwidth = frame.shape[:2]
        finally:
            capture.release()

        raw_imgs_dir = out_dir / video_name / "raw_imgs"
        raw_imgs_dir.mkdir(parents=True, exist_ok=True)

        files_csv_path = out_dir / video_name / "files.csv"
        with files_csv_path.open("w", newline="") as csv_out_file:
            csvwriter = csv.writer(
                csv_out_file, delimiter=",", quotechar='"', quoting=csv.QUOTE_MINIMAL
            )
            # Add new 'category' column. Extracted frames are 'original'.
            csvwriter.writerow(["frame", "file_loc", "stereo_partner", "upper_side", "video", "category"]) 

            capture = cv2.VideoCapture(str(video_path))

            image_count = 0
            frame_number = 0
            success = True

            try:
                if not capture.isOpened():
                    raise ValueError(f"couldn't open video: {video_path}")
                while capture.isOpened() and success:
                    success, frame = capture.read()
                    if not success:
                        break

                    # update status so we can resume if interrupted
                    json_index['status'] = {"step": "extract", "video": video_name, "frame": frame_number}
                    with json_out_path.open('w') as jf:
                        json.dump(json_index, jf, indent=2)

                    if frame_number < start_frame and video_name == start_video:
                        frame_number += 1
                        continue

                    if any(
                        frame_number >= sequence[0] and frame_number <= sequence[1]
                        for sequence in video_2_frame_number[video_name]["sequences"]
                    ):
                        filename = f"{video_name}_{frame_number}.png"
                        abs_file_path = raw_imgs_dir / filename
                        cv2.imwrite(str(abs_file_path), frame)

                        stereo_partners = []
                        for v in video_2_frame_number:
                            if v == video_name:
                                continue
                            if any(
                                frame_number >= sequence[0] and frame_number <= sequence[1]
                                for sequence in video_2_frame_number[v]["sequences"]
                            ):
                                stereo_partners.append(v)

                        # write original frame row and set category to 'original'
                        csvwriter.writerow(
                            [frame_number, str(abs_file_path), stereo_partners, "-", video_name, "original"]
                        )
                        image_count += 1

                        print(f"frame out: {frame_number}, total image count: {image_count}",end="\r")

                    frame_number += 1

                print(f"total image: {image_count}, done")

                json_index["frame_folders"] = list(dict.fromkeys(json_index.get("frame_folders", []) + [video_name]))
                json_index["index_files"][video_name] = str(files_csv_path)
                json_index["image_sizes"][video_name] = [int(vwidth), int(vheight)]
                json_index["image_count"] = json_index.get("image_count", 0) + image_count
                # extraction for this video finished
                json_index['status'] = {"step": "extract_complete", "video": video_name, "frame": frame_number}
                with json_out_path.open('w') as jf:
                    json.dump(json_index, jf, indent=2)
            finally:
                capture.release()

    # extraction all videos completed
    json_index['status'] = {"step": "extract_all_complete"}
    with json_out_path.open('w') as jf:
        json.dump(json_index, jf, indent=2)

    print('Extraction complete. index.json updated at', json_out_path)


In [ ]:
"""
Identify the upwards facing side based on two criteria:
    1) It has to include the top-most corner
    2) It has to be close to horizontal
"""
def identify_upwards_facing_edge(key_corners):
    neighbours_by_index = {
        0: [key_corners[1],key_corners[2]],
        1: [key_corners[0],key_corners[3]],
        2: [key_corners[0],key_corners[3]],
        3: [key_corners[1],key_corners[2]]
    }

    # Sort by Y coordinate
    sorted_by_y = sorted(key_corners, key=lambda p: p[1])[::-1] # opencv y-coordinates are inverted
    top_most = sorted_by_y[2:4]  # Two highest corners
    top_most_corner = top_most[1]
    top_most_index = next(i for i, c in enumerate(key_corners) if np.array_equal(c, top_most_corner))

    # each edge that includes the top-most corner is a candidate for the upwards-facing side.
    facing_up_candidates = []
    if top_most[1][1] != top_most[0][1]: # if there is exactly one top most corner, i.e. the upper side is not horizontal:
        for neighbour in neighbours_by_index[top_most_index]:
            facing_up_candidates.append((top_most_corner, neighbour))
    else: # in the unlikely case that the top two corners share their y-coordinate
        facing_up_candidates.append((top_most[1], top_most[0]))
    
    def compute_angle(p1, p2):
        return np.degrees(np.arctan2(p2[1] - p1[1], p2[0] - p1[0]))

    minDeviation = 90  # Start with the worst possible case
    facing_up = None
    for candidate in facing_up_candidates:
        angle = compute_angle(*candidate)
        deviation = min(abs(angle), abs(180 - abs(angle)))  # Ensure we measure closeness to horizontal
        if deviation < minDeviation:
            facing_up = candidate
            minDeviation = deviation

    return facing_up


"""
The (arbitrarily chosen) corner-naming for the checkerboard looks like this:
A         B
  |-|-|-|
  | | | |
  | | | |
D |-|-|-| C
  loopbio

Based on coordinates of the imagepoints and based on a user-given ground-truth about positions of corners in the image, 
a correspondence between the above real-world corners and imgpoints is recovered.
"""
def detect_corner_order(imgpoints, cbwidth, cbheight, upwards_facing_side):
    # Extract four key corners
    first = imgpoints[0]
    second = imgpoints[cbheight - 1]
    third = imgpoints[cbheight * cbwidth - cbheight]
    forth = imgpoints[cbheight * cbwidth - 1]
    key_corners = np.array([first, second, third, forth])
    
    facing_up = identify_upwards_facing_edge(key_corners)
    facing_up_left  = facing_up[0] if facing_up[0][0] < facing_up[1][0] else facing_up[1]
    facing_up_right = facing_up[0] if facing_up[0][0] > facing_up[1][0] else facing_up[1]
    
    # Now, we have a correspondence between the key-corners and real-world chessboard points
    # Identify the corresponding index
    index_of_facing_up_left_end = next(
        i for i, c in enumerate(key_corners) if np.all(np.isclose(c, facing_up_left))
    )
    index_of_facing_up_right_end = next(
        i for i, c in enumerate(key_corners) if np.all(np.isclose(c, facing_up_right))
    )

    print(f"facing up left corner index: {index_of_facing_up_left_end} ({facing_up_left})\nfacing up right corner index: {index_of_facing_up_right_end} ({facing_up_right})\n")

    # Find out which of the 8 corner orders is the order of the currently detected imagepoints based on where in the order the upwards pointing corners are:
    corner_order = None
    for short, order in complete_corner_mapping.items():
        if order[index_of_facing_up_left_end] == upwards_facing_side[0] and order[index_of_facing_up_right_end] == upwards_facing_side[1]:
            corner_order = short
            
    return corner_order
    

"""
A matrix rotation of 90 degree is a transpose with reordering.
Elements that were in a row have to be into a column after rotation, hence transpose.
If we then reverse the order within the rows (i.e. we push around columns) we get each former row (now column) to it's desired rotated position.
"""
def rotate_90_clockwise(m, iterations=1):
    for i in range(iterations):
        m = np.flip(m.transpose([1,0,2]), 0)
    return m

In [79]:
def reorder_corners(outdir, video_name, cbrows, cbcols, start_row: int = 0):
    """Reorder detected imgpoints based on upper_side labels from files.csv.
    Updates files.csv with confirmed upper_side values after reordering.
    Returns processed obj/imgpoints lists ready for calibration.
    Can resume from start_row index in files.csv.
    """
    outdir = Path(outdir)
    
    # Load index and files.csv
    with (outdir / 'index.json').open() as f:
        index = json.load(f)
    
    files_csv = Path(index['index_files'][video_name])
    files_data = []
    with files_csv.open() as f:
        reader = csv.DictReader(f)
        files_data = list(reader)

    # Load imgpoints
    imgpoints_path = Path(index['imgpoints'][video_name])
    with imgpoints_path.open() as f:
        imgpoints = json.load(f)

    # prepare object points template
    objp = np.zeros((cbrows*cbcols,3), np.float32)
    objp[:,:2] = np.mgrid[0:cbrows,0:cbcols].T.reshape(-1,2)

    # Create frame_number -> row_idx mapping for updates
    frame_to_row = {int(row['frame']): i for i, row in enumerate(files_data)}

    processed_objpoints = []
    processed_imgpoints = []

    # Process each frame in files.csv order, starting at start_row
    for idx in range(start_row, len(files_data)):
        row = files_data[idx]
        filename = os.path.splitext(os.path.basename(row['file_loc']))[0]
        if filename not in imgpoints:
            continue  # skip frames where no corners were detected
        
        points = np.array(imgpoints[filename])
        upper_side = row.get('upper_side', '-')
        
        # If no upper_side label, include points as-is for calibration
        if upper_side == '-' or not upper_side:
            imgp_ordered = points.reshape(cbrows*cbcols, 2)
        else:
            # detect current corner order and rotate/reorient as needed
            this_corner_order = detect_corner_order(points, cbheight=cbrows, cbwidth=cbcols, upwards_facing_side=upper_side)
            imgpt2 = np.copy(points).reshape(cbrows,cbcols,2)
            if this_corner_order is not None:
                if this_corner_order not in clockwise_rotation:
                    imgpt2 = np.flip(imgpt2, 1)
                    this_corner_order = this_corner_order[::-1]
                while this_corner_order != default_corner_order:
                    index_rot = clockwise_rotation.index(this_corner_order)
                    imgpt2 = rotate_90_clockwise(imgpt2, iterations=1)
                    index_rot = (index_rot-1) % len(clockwise_rotation)
                    this_corner_order = clockwise_rotation[index_rot]
                
                # Confirm the upper_side in files.csv now that we've validated it works
                frame_num = int(row['frame'])
                if frame_num in frame_to_row:
                    files_data[frame_to_row[frame_num]]['upper_side'] = upper_side
            
            imgp_ordered = imgpt2.reshape(cbrows*cbcols,2)

        processed_objpoints.append(objp.tolist())
        processed_imgpoints.append(imgp_ordered.tolist())

        # update status in index.json so pipeline can resume mid-reorder
        index['status'] = {"step": "reorder", "video": video_name, "row": idx, "frame": int(row['frame'])}
        with (outdir / 'index.json').open('w') as f:
            json.dump(index, f, indent=2)

    # Save processed points
    video_dir = outdir / video_name
    proc_objpoints_path = video_dir / 'processed_objpoints.json'
    proc_imgpoints_path = video_dir / 'processed_imgpoints.json'
    
    with proc_objpoints_path.open('w') as f:
        json.dump(processed_objpoints, f)
    with proc_imgpoints_path.open('w') as f:
        json.dump(processed_imgpoints, f)

    # Update files.csv with confirmed labels
    if len(files_data) > 0:
        with files_csv.open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=files_data[0].keys())
            writer.writeheader()
            writer.writerows(files_data)

    # Update index.json to track processed points
    if 'imgpoints_processed' not in index:
        index['imgpoints_processed'] = {}
    index['imgpoints_processed'][video_name] = {
        'objpoints': str(proc_objpoints_path),
        'imgpoints': str(proc_imgpoints_path)
    }
    index['status'] = {"step": "reorder_complete", "video": video_name}
    with (outdir / 'index.json').open('w') as f:
        json.dump(index, f, indent=2)

    return processed_objpoints, processed_imgpoints

## Checkerboard Detection

In [80]:
def detect_checkerboards(outdir, video_name, cbrows, cbcols, enhance=True, slope_change_threshold_deg=15, start_row: int = 0):
    """Detect checkerboards in images from files.csv, track results in index.json.
    Uses files.csv for frame numbers and updates it with upper_side labels.
    Can resume from a given row index in files.csv by passing start_row.
    Marks frames that need user labeling by setting upper_side to "!" in files.csv and
    adds 'cb_detected' category entries to files.csv for visualizations created.
    """
    outdir = Path(outdir)
    index_path = outdir / 'index.json'
    with index_path.open() as f:
        index = json.load(f)

    # Load files.csv for this video
    files_csv = Path(index['index_files'][video_name])
    files_data = []
    with files_csv.open() as f:
        reader = csv.DictReader(f)
        files_data = list(reader)

    # Ensure 'category' column exists; default missing values to 'original'
    for r in files_data:
        if 'category' not in r or not r['category']:
            r['category'] = 'original'

    # Build a queue: original rows that do not yet have a corresponding 'cb_detected' entry
    existing_cb_basenames = set()
    for r in files_data:
        if r.get('category') == 'cb_detected':
            existing_cb_basenames.add(os.path.basename(r['file_loc']))

    original_rows = [ (i, r) for i, r in enumerate(files_data) if r.get('category') == 'original' ]
    queue = []
    for i, r in original_rows:
        orig_basename = os.path.basename(r['file_loc'])
        if orig_basename not in existing_cb_basenames:
            queue.append((i, r))

    # criteria for corner detection
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

    # prepare object points template
    objp = np.zeros((cbcols*cbrows,3), np.float32)
    objp[:,:2] = np.mgrid[0:cbrows,0:cbcols].T.reshape(-1,2)

    # Arrays to store object points and image points from all the images.
    objpoints = {} # 3d point in real world space
    imgpoints = {} # 2d points in image plane.

    prev_angle = None
    changed_frames = []  # list of (filename, frame_number, row_idx)

    video_dir = outdir / video_name
    vis_dir = video_dir / 'visualizations'
    vis_dir.mkdir(parents=True, exist_ok=True)

    print(f"Found {len(queue)} original frames that need cb detection (rows in files.csv).")

    # Process the queue (only entries missing cb_detected)
    for i, row in queue:
        fpath = row['file_loc']
        frame_num = int(row['frame'])
        filename = os.path.splitext(os.path.basename(fpath))[0]

        # update status
        index['status'] = {"step": "detect", "video": video_name, "frame": frame_num, "row": i}
        with index_path.open('w') as jf:
            json.dump(index, jf, indent=2)

        img = cv2.imread(fpath)
        if img is None:
            print(f"Could not read {fpath}; skipping")
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        if enhance:
            gray = enhance_image(gray)

        try:
            ret, corners = cv2.findChessboardCornersSB(gray, (cbrows,cbcols), None)
        except Exception:
            ret, corners = cv2.findChessboardCorners(gray, (cbrows,cbcols), None)

        if not ret:
            # No corners found: skip creating cb_detected entry but keep original row
            continue

        # corners found: refine and save
        try:
            corners2 = cv2.cornerSubPix(gray,corners, (cbrows-1,cbcols-1), (-1,-1), criteria)
        except Exception:
            corners2 = corners

        objpoints[filename] = objp.tolist()
        imgpoints[filename] = corners2.tolist()
        imgpoints[filename] = [point[0] for point in imgpoints[filename]] # remove one unnecessary dimension in the array (points were unnecessarily stored as [[x,y]] instead of [x,y])

        # Compute edge orientation and mark '!' on the original row if angle changes
        first = imgpoints[filename][0]
        second = imgpoints[filename][cbrows - 1]
        third = imgpoints[filename][cbrows * cbcols - cbrows]
        forth = imgpoints[filename][cbrows * cbcols - 1]
        key_corners = np.array([first, second, third, forth])

        facing_up = identify_upwards_facing_edge(key_corners)
        p1, p2 = map(tuple, map(np.int32, facing_up))
        try:
            angle_deg = np.degrees(np.arctan2(facing_up[1][1] - facing_up[0][1], facing_up[1][0] - facing_up[0][0]))
        except Exception:
            angle_deg = None

        if angle_deg is not None and prev_angle is not None:
            diff = abs(angle_deg - prev_angle)
            diff = min(diff, 360 - diff)
            if diff > slope_change_threshold_deg:
                # mark the ORIGINAL row for manual labeling
                files_data[i]['upper_side'] = '!'
                changed_frames.append((os.path.basename(fpath), frame_num, i))
        if angle_deg is not None:
            prev_angle = angle_deg

        # create visualization
        color_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) if len(gray.shape) == 2 else img.copy()
        cv2.line(color_img, p1, p2, (0, 255, 0), 2)
        first_corner = (int(corners2[0][0][0]), int(corners2[0][0][1]))
        last_corner = (int(corners2[-1][0][0]), int(corners2[-1][0][1]))
        cv2.putText(color_img, '0', first_corner, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(color_img, str(len(corners2) - 1), last_corner, cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
        color_img = draw_corners_with_order(color_img, corners2)

        vis_name = os.path.basename(fpath)
        vis_path = vis_dir / vis_name
        cv2.imwrite(str(vis_path), color_img)

        # Append a new files.csv entry for this detection with category 'cb_detected'
        new_row = {
            'frame': row['frame'],
            'file_loc': str(vis_path),
            'stereo_partner': row.get('stereo_partner', '[]'),
            'upper_side': row.get('upper_side', '-') ,
            'video': video_name,
            'category': 'cb_detected'
        }
        files_data.append(new_row)

    # After processing queue, save detected points and write files.csv
    video_dir.mkdir(exist_ok=True)
    objpoints_path = video_dir / 'objpoints.json'
    imgpoints_path = video_dir / 'imgpoints.json'
    with objpoints_path.open('w') as f:
        json.dump(objpoints, f)
    with imgpoints_path.open('w') as f:
        json.dump(imgpoints, f)

    # Ensure header includes 'category'
    if len(files_data) > 0:
        with files_csv.open('w', newline='') as f:
            fieldnames = list(files_data[0].keys())
            # ensure consistent ordering
            if 'category' not in fieldnames:
                fieldnames.append('category')
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(files_data)

    # Update index.json to track imgpoints and mark that corner-correction may be pending
    if 'imgpoints' not in index:
        index['imgpoints'] = {}
    index['imgpoints'][video_name] = str(imgpoints_path)
    if len(changed_frames) > 0:
        index['status'] = {"step": "corner_correction_pending", "video": video_name, "count": len(changed_frames)}
    else:
        index['status'] = {"step": "detect_complete", "video": video_name}
    with index_path.open('w') as jf:
        json.dump(index, jf, indent=2)

    return objpoints, imgpoints

In [81]:
def label_upper_side(outdir, video_name, start_row: int = 0):
    """Interactive widget-based correction step: show frames marked with '!' in files.csv and ask user to assign upper_side.

    Behavior:
      - Reads files.csv for the given video and finds rows where upper_side == '!'.
      - Displays each marked frame in a widget with image + text input field.
      - User enters upwards-facing side (e.g. 'BC') and clicks "Next" button.
      - Label is assigned to the marked row and propagated forward until the next '!' marker or EOF.
      - Updates files.csv and index.json status so the pipeline can resume.
      - Supports resuming by honoring start_row and updating index['status'] as it progresses.
    """
    outdir = Path(outdir)
    index_path = outdir / 'index.json'
    with index_path.open() as f:
        index = json.load(f)

    files_csv = Path(index['index_files'][video_name])
    with files_csv.open() as f:
        reader = csv.DictReader(f)
        files_data = list(reader)

    video_dir = outdir / video_name
    vis_dir = video_dir / 'visualizations'

    # Gather indices of rows that are marked as needing correction
    marked_rows = [i for i, r in enumerate(files_data) if r.get('upper_side', '') == '!']
    # Filter by start_row to support resume
    marked_rows = [r for r in marked_rows if r >= start_row]

    if len(marked_rows) == 0:
        print(f"No frames marked for corner-order correction for {video_name} (start_row={start_row}).")
        # Update status to indicate nothing to do
        index['status'] = {"step": "corner_correction_complete", "video": video_name}
        with index_path.open('w') as jf:
            json.dump(index, jf, indent=2)
        return files_csv

    print(f"\nFound {len(marked_rows)} frames marked for correction (starting at row {start_row}).")

    complete_corner_mapping_set = {"AB", "BC", "CD", "DA", "BA", "CB", "DC", "AD"}

    # State dict to track current marking index and collected data
    state = {
        'current_idx': 0,
        'user_input': '',
        'done': False,
    }

    # Container for the interactive UI
    output_area = widgets.Output()
    
    def update_ui():
        """Render the current frame and input controls."""
        output_area.clear_output(wait=True)
        
        if state['current_idx'] >= len(marked_rows):
            with output_area:
                print("✓ All frames labeled! Finalizing...")
            finalize()
            return

        idx = marked_rows[state['current_idx']]
        row = files_data[idx]
        frame_num = int(row['frame'])
        
        with output_area:
            print(f"\n{'='*60}")
            print(f"Frame {frame_num} ({state['current_idx'] + 1}/{len(marked_rows)})")
            print(f"{'='*60}")
            
            # Display the visualization image
            vis_path = vis_dir / os.path.basename(row['file_loc'])
            if vis_path.exists():
                display(IPImage(filename=str(vis_path)))
            else:
                print(f"⚠ Visualization not found: {vis_path}")
                imgfile = Path(row['file_loc'])
                if imgfile.exists():
                    display(IPImage(filename=str(imgfile)))

    def on_submit(change):
        """Handle user input and move to next frame."""
        user_input = text_input.value.strip().upper()
        
        if len(user_input) == 0:
            user_input = "BC"
        if len(user_input) > 2:
            user_input = user_input[:2]
        
        # Validate and warn if not standard
        if user_input not in complete_corner_mapping_set:
            print(f"⚠ Warning: '{user_input}' not standard, but saving anyway.")
        
        # Get current marked row index
        idx = marked_rows[state['current_idx']]
        
        # Find next marked row after this one
        next_mark = None
        for m in marked_rows:
            if m > idx:
                next_mark = m
                break
        end_idx = next_mark if next_mark is not None else len(files_data)
        
        # Assign label to all rows from idx to end_idx
        for r in range(idx, end_idx):
            files_data[r]['upper_side'] = user_input
        
        # Persist immediately
        with files_csv.open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=files_data[0].keys())
            writer.writeheader()
            writer.writerows(files_data)
        
        # Update index status
        index['status'] = {"step": "corner_correction_in_progress", "video": video_name, "row": end_idx - 1}
        with index_path.open('w') as jf:
            json.dump(index, jf, indent=2)
        
        # Clear input and move to next
        text_input.value = ''
        state['current_idx'] += 1
        
        update_ui()

    def finalize():
        """Write final files and mark correction complete."""
        with files_csv.open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=files_data[0].keys())
            writer.writeheader()
            writer.writerows(files_data)
        
        index['status'] = {"step": "corner_correction_complete", "video": video_name}
        with index_path.open('w') as jf:
            json.dump(index, jf, indent=2)
        
        with output_area:
            print(f"\n✓ Corner-order correction complete for {video_name}.")
        state['done'] = True

    # Create widgets
    text_input = widgets.Text(
        value='',
        placeholder='Enter upwards-facing side (e.g. BC or AB)',
        description='Upper side:',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='400px')
    )
    
    next_button = widgets.Button(
        description='Next',
        button_style='info',
        tooltip='Submit input and move to next frame',
        icon='arrow-right'
    )
    
    instruction_label = widgets.HTML(
        "<b>Inspect the image and enter which side of the checkerboard points upward (e.g. 'BC').</b>"
    )
    
    # Wire up event handlers
    text_input.observe(on_submit, names='value')
    next_button.on_click(lambda b: on_submit(None))
    
    # Display the UI
    controls_box = widgets.VBox([
        instruction_label,
        output_area,
        text_input,
        next_button
    ])
    
    display(controls_box)
    
    # Render first frame
    update_ui()
    
    # Wait for completion (note: in Jupyter, this returns immediately but callbacks continue)
    # For blocking behavior, you'd need async/await, but widget callbacks are sufficient here.

    return files_csv

## Calibration

In [82]:
def calibrate_camera_from_processed(outdir, video_name, image_size):
    """Calibrate a single camera from processed points and update index.json with results."""
    outdir = Path(outdir)
    
    # Load index
    with (outdir / 'index.json').open() as f:
        index = json.load(f)

    # Get paths to processed points
    proc_paths = index['imgpoints_processed'][video_name]
    with open(proc_paths['objpoints']) as f:
        objpoints_list = json.load(f)
    with open(proc_paths['imgpoints']) as f:
        imgpoints_list = json.load(f)

    # Convert to numpy arrays
    objpoints = np.array(objpoints_list, dtype=np.float32)
    imgpoints = np.array(imgpoints_list, dtype=np.float32)

    # Run calibration
    ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, tuple(image_size), None, None, criteria=criteria)

    calibration_data = {
        "video_name": video_name,
        "ret": float(ret),
        "mtx": mtx.tolist(),
        "dist": dist.tolist(),
        "rvecs": [r.tolist() for r in rvecs],
        "tvecs": [t.tolist() for t in tvecs],
    }

    # Save calibration parameters
    calib_dir = outdir / 'calibration'
    calib_dir.mkdir(exist_ok=True)
    calib_path = calib_dir / f"{video_name}_calibration.json"
    
    with calib_path.open('w') as f:
        json.dump(calibration_data, f, indent=4)

    # Update index.json to track calibration file
    if 'calib' not in index:
        index['calib'] = {}
    index['calib'][video_name] = str(calib_path)
    
    with (outdir / 'index.json').open('w') as f:
        json.dump(index, f, indent=2)

    print(f"Saved calibration for {video_name} to {calib_path}")
    return calibration_data

def stereo_calibrate(outdir, video1, video2):
    """Run stereo calibration for a pair of videos and track results in index.json."""
    outdir = Path(outdir)
    
    # Load index
    with (outdir / 'index.json').open() as f:
        index = json.load(f)

    # Load calibration parameters
    def load_calib(video):
        with open(index['calib'][video]) as f:
            return json.load(f)
    
    calib1 = load_calib(video1)
    calib2 = load_calib(video2)

    # Get processed points
    with open(index['imgpoints_processed'][video1]['objpoints']) as f:
        objpoints = np.array(json.load(f), dtype=np.float32)
    with open(index['imgpoints_processed'][video1]['imgpoints']) as f:
        imgpoints1 = np.array(json.load(f), dtype=np.float32)
    with open(index['imgpoints_processed'][video2]['imgpoints']) as f:
        imgpoints2 = np.array(json.load(f), dtype=np.float32)

    # Get image sizes
    size1 = np.array(index['image_sizes'][video1])
    size2 = np.array(index['image_sizes'][video2])

    # Extract intrinsics
    mtx1 = np.array(calib1['mtx'])
    dist1 = np.array(calib1['dist'])
    mtx2 = np.array(calib2['mtx'])
    dist2 = np.array(calib2['dist'])

    # Run stereo calibration
    retval, cameraMatrix1, distCoeffs1, cameraMatrix2, distCoeffs2, R, T, E, F = (
        cv2.stereoCalibrate(
            objpoints, imgpoints1, imgpoints2,
            mtx1, dist1, mtx2, dist2, size1[::-1],
            flags=cv2.CALIB_FIX_INTRINSIC,
        )
    )

    # Fix rotation direction
    R_flipX = cv2.Rodrigues(np.array([np.pi, 0, 0]))[0]
    R_fixed = R @ R_flipX
    R = R_fixed
    T = -T

    # Plot relationship
    r = scipy.spatial.transform.Rotation.from_matrix(R)
    angles_deg = r.as_euler("xyz", degrees=True)
    print("Rotation (degrees):", angles_deg)
    print("Translation vector:", T.ravel())

    # === Plot camera setup ===
    C1 = np.array([0, 0, 0])
    R1 = np.eye(3)
    C2 = -R.T @ T
    R2 = R

    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")
    draw_camera(ax, C1, R1, color="blue", label=f"Camera {video1}")
    draw_camera(ax, C2.ravel(), R2, color="red", label=f"Camera {video2}")
    ax.plot([C1[0], C2[0][0]], [C1[1], C2[1][0]], [C1[2], C2[2][0]], "k--", label="Baseline")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(f"Stereo Setup: {video1} - {video2}")
    ax.legend()
    ax.view_init(elev=20, azim=45)
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    # Calculate projection matrices
    K1 = cameraMatrix1
    K2 = cameraMatrix2
    R1 = np.eye(3)
    t1 = np.zeros((3, 1))
    P1 = K1 @ np.hstack((R1, t1))
    P2 = K2 @ np.hstack((R2, T))

    stereo_params = {
        "video1": video1,
        "video2": video2,
        "cameraMatrix1": cameraMatrix1.tolist(),
        "distCoeffs1": distCoeffs1.tolist(),
        "cameraMatrix2": cameraMatrix2.tolist(),
        "distCoeffs2": distCoeffs2.tolist(),
        "R": R.tolist(),
        "T": T.tolist(),
        "E": E.tolist(),
        "F": F.tolist(),
        "P1": P1.tolist(),
        "P2": P2.tolist(),
    }

    # Save stereo parameters
    stereo_dir = outdir / 'stereo'
    stereo_dir.mkdir(exist_ok=True)
    stereo_path = stereo_dir / f"{video1}__{video2}_stereo.json"
    
    with stereo_path.open('w') as f:
        json.dump(stereo_params, f, indent=4)

    # Update index.json
    if 'stereo_calib' not in index:
        index['stereo_calib'] = {}
    index['stereo_calib'][f"{video1}__{video2}"] = str(stereo_path)
    
    with (outdir / 'index.json').open('w') as f:
        json.dump(index, f, indent=2)

    print(f"Stereo calibration saved to {stereo_path}")
    return stereo_params

In [83]:
def stereo_calibrate(objpoints, imgpoints1, imgpoints2, img1_shape_np, img2_shape_np, output_dir):
    ret1, mtx1, dist1, rvecs1, tvecs1 = cv2.calibrateCamera(
        objpoints, imgpoints1, img1_shape_np[::-1], None, None
    )
    ret2, mtx2, dist2, rvecs2, tvecs2 = cv2.calibrateCamera(
        objpoints, imgpoints2, img2_shape_np[::-1], None, None
    )

    retval, cameraMatrix1, distCoeffs1, cameraMatrix2, distCoeffs2, R, T, E, F = (
        cv2.stereoCalibrate(
            objpoints,
            imgpoints1,
            imgpoints2,
            mtx1,
            dist1,
            mtx2,
            dist2,
            img1_shape_np[::-1],
            flags=cv2.CALIB_FIX_INTRINSIC,
        )
    )

    # fix rotation direction
    R_flipX = cv2.Rodrigues(np.array([np.pi, 0, 0]))[0]
    R_fixed = R @ R_flipX

    R = R_fixed
    T = -T

    # plot relationship
    r = scipy.spatial.transform.Rotation.from_matrix(R)
    angles_deg = r.as_euler("xyz", degrees=True)
    print("Rotation (degrees):", angles_deg)
    print("Translation vector:", T.ravel())

    # Camera 1 (origin)
    C1 = np.array([0, 0, 0])
    R1 = np.eye(3)
    t1 = np.zeros((3, 1))

    # Camera 2
    R2 = R
    C2 = -R2.T @ T  # Compute camera 2 center in world coords

    # === Plot both cameras ===
    fig = plt.figure()
    ax = fig.add_subplot(111, projection="3d")

    draw_camera(ax, C1, R1, color="blue", label="Camera 1")
    draw_camera(ax, C2.ravel(), R2, color="red", label="Camera 2")

    # Draw baseline
    ax.plot(
        [C1[0], C2[0][0]],
        [C1[1], C2[1][0]],
        [C1[2], C2[2][0]],
        "k--",
        label="Baseline",
    )

    # Axes settings
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("Stereo Camera Relationship")
    ax.legend()
    ax.view_init(elev=20, azim=45)
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    # rectify_scale = alpha
    # R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
    #     cameraMatrix1, distCoeffs1, cameraMatrix2, distCoeffs2, enhanced_gray1.shape[::-1], R, T, alpha=rectify_scale
    # )

    # calculate stereo matrix
    # Intrinsics (from calibration)
    K1 = cameraMatrix1  # 3x3
    K2 = cameraMatrix2  # 3x3

    # Camera 1 at origin
    R1 = np.eye(3)
    t1 = np.zeros((3, 1))
    P1 = K1 @ np.hstack((R1, t1))  # 3x4

    # Camera 2 positioned relative to camera 1
    P2 = K2 @ np.hstack((R2, T))

    stereo_params = {
        "cameraMatrix1": cameraMatrix1,
        "distCoeffs1": distCoeffs1,
        "cameraMatrix2": cameraMatrix2,
        "distCoeffs2": distCoeffs2,
        "R": R,
        "T": T,
        "E": E,
        "F": F,
        "R1": R1,
        "R2": R2,
        "P1": P1,
        "P2": P2,
        # 'Q': Q,
        # 'roi1': roi1,
        # 'roi2': roi2
    }

    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "stereo_params.pickle"), "wb") as f:
        pickle.dump(stereo_params, f)

    print(
        f"Stereo calibration complete. Parameters saved to {output_dir}/stereo_params.pickle"
    )

## Usage

In [84]:
def calibrate_videos(videos, video_2_frame_number, outdir, cbrows, cbcols, slope_change_threshold_deg=15):
    """Full calibration pipeline that can resume using index.json status.

    Behavior:
      - If index.json exists, read index['status'] and resume at that step/video/frame.
      - Steps: extract -> detect -> corner_correction -> reorder -> calibrate -> stereo
    """
    outdir = Path(outdir)
    videos = [Path(v) for v in videos]

    index_path = outdir / 'index.json'
    existing_index = None
    if index_path.exists():
        with index_path.open() as f:
            existing_index = json.load(f)

    # Determine resume point
    resume = None
    resume_video = None
    resume_frame = 0
    resume_row = 0
    if existing_index and 'status' in existing_index:
        status = existing_index['status']
        resume = status.get('step')
        resume_video = status.get('video')
        resume_frame = status.get('frame', 0)
        resume_row = status.get('row', 0)
        print(f"Resuming pipeline from status: {status}")

    # 1) Extract frames (resume-aware)
    if resume in (None, 'extract', 'extract_complete', 'extract_all_complete'):
        # If we have an existing status and it's 'extract', pass start_video/frame
        start_video = resume_video if resume == 'extract' else None
        start_frame = resume_frame if resume == 'extract' else 0
        extract_from_video(videos, video_2_frame_number, outdir, start_video=start_video, start_frame=start_frame)
    else:
        print('Skipping extraction (already done or resuming later)')

    # Reload index after extraction
    with index_path.open() as f:
        index = json.load(f)
    video_names = index.get('frame_folders', [])

    # 2) Detect checkerboards (resume-aware)
    for vname in video_names:
        if resume == 'detect' and resume_video == vname:
            # resume from recorded row
            start_row = resume_row
            print(f"Resuming detect for {vname} at row {start_row}")
            detect_checkerboards(str(outdir), vname, cbrows, cbcols, enhance=True, slope_change_threshold_deg=slope_change_threshold_deg, start_row=start_row)
            resume = None
        else:
            print(f"Running detect for {vname}")
            detect_checkerboards(str(outdir), vname, cbrows, cbcols, enhance=True, slope_change_threshold_deg=slope_change_threshold_deg, start_row=0)

    # Reload index after detection
    with index_path.open() as f:
        index = json.load(f)

    # 2.5) Corner-order correction step (interactive) - resume-aware
    for vname in video_names:
        if resume == 'corner_correction' and resume_video == vname:
            start_row = resume_row
            print(f"Resuming corner-order correction for {vname} at row {start_row}")
            label_upper_side(str(outdir), vname, start_row=start_row)
            resume = None
        else:
            # Only run correction if detect marked any frames or if explicitly resuming
            # Check index to see if this video had pending corrections
            pending = False
            try:
                with (outdir / 'index.json').open() as jf:
                    idx = json.load(jf)
                    if idx.get('status', {}).get('step') in ('corner_correction_pending', 'corner_correction') and idx.get('status', {}).get('video') == vname:
                        pending = True
            except Exception:
                pending = False

            if pending:
                print(f"Running corner-order correction for {vname}")
                label_upper_side(str(outdir), vname, start_row=0)
            else:
                print(f"No corner-order correction needed for {vname}, skipping")

    # Reload index after corner correction
    with index_path.open() as f:
        index = json.load(f)

    # 3) Reorder corners (resume-aware)
    for vname in video_names:
        if resume == 'reorder' and resume_video == vname:
            start_row = resume_row
            print(f"Resuming reorder for {vname} at row {start_row}")
            reorder_corners(str(outdir), vname, cbrows, cbcols, start_row=start_row)
            resume = None
        else:
            print(f"Running reorder for {vname}")
            reorder_corners(str(outdir), vname, cbrows, cbcols, start_row=0)

    # Reload index after reorder
    with index_path.open() as f:
        index = json.load(f)

    # 4) Run per-video calibration
    for vname in video_names:
        # skip if already calibrated (index tracks calib)
        if 'calib' in index and vname in index['calib']:
            print(f"Calibration already present for {vname}, skipping")
            continue
        image_size = index.get('image_sizes', {}).get(vname)
        if image_size is None:
            print(f"No image size for {vname}; skipping")
            continue
        calibrate_camera_from_processed(str(outdir), vname, image_size)

    # 5) Run stereo calibration for each pair
    from itertools import combinations
    for v1, v2 in combinations(video_names, 2):
        # skip if stereo already exists
        with index_path.open() as f:
            index = json.load(f)
        key = f"{v1}__{v2}"
        if 'stereo_calib' in index and key in index['stereo_calib']:
            print(f"Stereo calib for {v1}-{v2} already present, skipping")
            continue
        print(f"Running stereo calibration: {v1} - {v2}")
        stereo_calibrate(str(outdir), v1, v2)

    # Final status update
    with index_path.open() as f:
        index = json.load(f)
    index['status'] = {"step": "pipeline_complete"}
    with index_path.open('w') as f:
        json.dump(index, f, indent=2)

    print("\nCalibration pipeline complete. Index updated.")

In [85]:
videos = [
    '/media/jonathan/library/fish_data/bluegill_calib/cam-1/cam-1_15-17-14-calibration.avi',
    '/media/jonathan/library/fish_data/bluegill_calib/cam-2/cam-2_15-17-14-calibration.avi',
    '/media/jonathan/library/fish_data/bluegill_calib/cam-3/cam-3_15-17-14-calibration.avi',
    '/media/jonathan/library/fish_data/bluegill_calib/cam-4/cam-4_15-17-14-calibration.avi'
]
video_2_frame_number = '/media/jonathan/library/fish_data/bluegill_calib/video_2_frame_number.json'
outdir = '/home/jonathan/Documents/fish_reconstruction/calibration/calib_outputs'
cbrows, cbcols = 8, 9  # inner corner counts used for detection/calibration

calibrate_videos(videos, video_2_frame_number, outdir, cbrows, cbcols)

Resuming pipeline from status: {'step': 'detect', 'video': 'cam-1_15-17-14-calibration', 'frame': 627, 'row': 13}
Skipping extraction (already done or resuming later)
Resuming detect for cam-1_15-17-14-calibration at row 13
Found 1056 original frames that need cb detection (rows in files.csv).
[[ 454.89077759 1081.0682373 ]
 [ 558.11193848  880.77581787]
 [ 645.73156738 1199.39428711]
 [ 754.0333252   992.65838623]]
[[ 450.0362854  1082.73083496]
 [ 552.17346191  880.76385498]
 [ 652.50256348 1196.49597168]
 [ 758.7088623   987.7409668 ]]
[[ 445.9543457  1083.03259277]
 [ 546.64624023  879.45178223]
 [ 658.18182373 1193.47961426]
 [ 762.26269531  983.67565918]]
[[ 443.04537964 1083.53234863]
 [ 541.26721191  878.11425781]
 [ 662.3571167  1189.80627441]
 [ 763.53796387  978.7144165 ]]
[[ 440.82064819 1082.53601074]
 [ 537.49505615  876.15820312]
 [ 665.05426025 1185.45263672]
 [ 763.82000732  973.44769287]]
[[ 439.09442139 1081.21728516]
 [ 533.91992188  874.17376709]
 [ 666.96179199 11

KeyboardInterrupt: 